In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score

In [ ]:
!pip install xgboost

In [2]:
def split_scalar(indep_x1,dep_y1):    
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    sc=StandardScaler()
    x_train,x_test,y_train,y_test=train_test_split(indep_x1,dep_y1,test_size=0.30,random_state=0)
    x_train_pp=sc.fit_transform(x_train)
    x_test_pp=sc.transform(x_test)
    return x_train_pp,x_test_pp,y_train, y_test,sc

def mlr(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.linear_model import LinearRegression
    reg=LinearRegression()
    reg.fit(x_train_pp,y_train)
    y_pred = reg.predict(x_test_pp)
    re=r2_score(y_test,y_pred)
    return re,reg

def svm(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.svm import SVR
    param_grid={'kernel':['rbf','poly','sigmoid','linear'],'C':[10,1000,1000,2000,3000]}
    grid=GridSearchCV(SVR(),param_grid,refit=True,verbose=3,n_jobs=-1)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_

def dt(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.tree import DecisionTreeRegressor
    param_grid = {'criterion':    ['squared_error', 'absolute_error', 'friedman_mse'],  # ✅ fixed
        'max_features': ['sqrt', 'log2', None],                               # ✅ fixed
        'splitter':     ['best', 'random']}
    grid = GridSearchCV(DecisionTreeRegressor(), param_grid, refit=True, cv=5, n_jobs=-1)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_
    
def rf(x_train_pp,x_test_pp,y_train,y_test):    
    from sklearn.ensemble import RandomForestRegressor
    param_grid = {'criterion':    ['squared_error', 'absolute_error'],                  # ✅ fixed
        'max_features': ['sqrt', 'log2', None],                               # ✅ fixed
        'n_estimators': [10, 100]}
    grid = GridSearchCV(RandomForestRegressor(), param_grid, refit=True, cv=5, n_jobs=-1)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_

def xg(x_train_pp,x_test_pp,y_train,y_test):    

    from xgboost import XGBRegressor
    param_grid = {'n_estimators':  [100, 200],'max_depth':[3, 5], 'learning_rate': [0.01, 0.1], 'subsample': [0.8, 1.0] }
    grid= GridSearchCV(XGBRegressor(random_state=42),param_grid,scoring='r2',cv=5,n_jobs=-1,refit=True)
    grid.fit(x_train_pp,y_train)
    y_pred = grid.predict(x_test_pp)
    re= r2_score(y_test, y_pred)
    return re,grid.best_estimator_


def Reg_rscore_result(regmlr,regsvm,regdt,regrf,regxg): 
    
    dataframe=pd.DataFrame(index=['rscore'],columns=['MLR','SVM','DecisionTree','RandomForest','XGBoosting'])

    for number,idex in enumerate(dataframe.index):
        
        dataframe.loc[idex, 'MLR']          = regmlr[number]
        dataframe.loc[idex, 'SVM']          = regsvm[number]
        dataframe.loc[idex, 'DecisionTree'] = regdt[number]
        dataframe.loc[idex, 'RandomForest'] = regrf[number]
        dataframe.loc[idex, 'XGBoosting']   = regxg[number]
    return dataframe








In [3]:
df = pd.read_csv('preprocessed_data.csv')
df

,age,screen_time_hours,work_screen_hours,leisure_screen_hours,sleep_hours,sleep_quality_1_5,stress_level_0_10,productivity_0_100,exercise_minutes_per_week,social_hours_per_week,mental_wellness_index_0_100,gender_Male,gender_Non-binary/Other,occupation_Retired,occupation_Self-employed,occupation_Student,occupation_Unemployed,work_mode_In-person,work_mode_Remote
0,33.0,10.0,5.0,5.0,6.0,1.0,9,44,127.0,0.0,9.0,0,0,0,0,0,0,0,1
1,28.0,7.0,0.0,7.0,8.0,3.0,5,78,74.0,2.0,56.0,0,0,0,0,0,0,1,0
2,35.0,9.0,1.0,8.0,6.0,1.0,9,51,67.0,8.0,3.0,0,0,0,0,0,0,0,0
3,42.0,11.0,0.0,10.0,6.0,1.0,10,37,0.0,5.0,0.0,1,0,0,0,0,0,0,0
4,28.0,13.0,4.0,9.0,5.0,1.0,10,38,143.0,10.0,0.0,1,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,26.0,6.0,2.0,3.0,7.0,1.0,5,64,252.0,7.0,39.0,0,0,0,0,1,0,0,1
396,16.0,9.0,5.0,4.0,5.0,1.0,10,47,99.0,7.0,3.0,1,0,0,0,0,0,0,1
397,40.0,8.0,2.0,6.0,7.0,1.0,9,57,193.0,10.0,6.0,1,0,0,0,1,0,0,1
398,29.0,5.0,0.0,4.0,7.0,1.0,7,63,97.0,12.0,21.0,0,0,0,0,0,0,0,0


In [4]:
indep_x=df[['screen_time_hours', 'sleep_quality_1_5', 'stress_level_0_10','productivity_0_100','exercise_minutes_per_week', 'social_hours_per_week']]
dep_y=df['mental_wellness_index_0_100']

In [5]:
indep_x1=indep_x
dep_y1=dep_y


In [6]:
relist=[1]
regmlr=[]
regsvm=[]
regdt=[]
regrf=[]
regxg=[]





In [7]:

for i in relist:   
    x_train_pp, x_test_pp, y_train, y_test,sc=split_scalar(indep_x,dep_y) 
    remlr,model_mlr=mlr(x_train_pp,x_test_pp,y_train,y_test)
    regmlr.append(remlr)
    
    resvm,model_svm=svm(x_train_pp,x_test_pp,y_train,y_test)
    regsvm.append(resvm)
    
    redt,model_dt=dt(x_train_pp,x_test_pp,y_train,y_test)
    regdt.append(redt)
    
    rerf,model_rf=rf(x_train_pp,x_test_pp,y_train,y_test)
    regrf.append(rerf)
    
    rexg,model_xg=xg(x_train_pp,x_test_pp,y_train,y_test)
    regxg.append(rexg)

result=Reg_rscore_result(regmlr,regsvm,regdt,regrf,regxg)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


In [8]:
result

,MLR,SVM,DecisionTree,RandomForest,XGBoosting
rscore,0.923616,0.921986,0.767942,0.922222,0.922966


In [9]:
resultMLR=remlr
resultMLR

0.9236162680758219

In [10]:
import pickle
# Step 2 — Create a dictionary of all models
all_models = {
    'MLR':          regmlr,
    'SVM':          regsvm,
    'DecisionTree': regdt,
    'RandomForest': regrf,
    'XGBoosting':   regxg}
# Step 3 — Automatically pick best model by highest r2 score
best_algo = result.loc['rscore'].astype(float).idxmax()
best_model = all_models[best_algo]
print(f"Best Algorithm: {best_algo}")

Best Algorithm: MLR


In [11]:
filename="screentime vs Mentalwellness_prepro X only.sav"

pickle.dump(sc,open(filename,'wb'))

In [12]:
input=pickle.load(open("screentime vs Mentalwellness_prepro X only.sav",'rb'))

In [14]:

filename1="screentime vs Mentalwellness_MLR.sav"

pickle.dump(model_mlr,open(filename1,'wb'))

In [15]:
loaded_model=pickle.load(open("screentime vs Mentalwellness_MLR.sav",'rb'))

In [16]:
print(type(loaded_model))

<class 'sklearn.linear_model._base.LinearRegression'>


In [17]:
preinput = pd.DataFrame([[9, 2, 5, 50, 200, 1]],
                         columns=['screen_time_hours', 'sleep_quality_1_5',
                                  'stress_level_0_10', 'productivity_0_100',
                                  'exercise_minutes_per_week', 'social_hours_per_week'])
preinput_scaled=sc.transform(preinput)
Mental_wellness_result=loaded_model.predict(preinput_scaled)
Mental_wellness_result

array([39.08282864])